<a href="https://colab.research.google.com/github/santiiis/proyectobigdata/blob/main/Proyecto_Final_BigData_Desercion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Proyecto Integrador: Predicción de deserción estudiantil
**Integrantes:** Lander González, Erick Morales  
**Tema:** Predicción de deserción estudiantil (Grupo 7)  
**Asignatura:** Prácticas y Herramientas de Big Data  


### 1. Inicialización de Sesión Distribuida en Apache Spark y Carga de Datos OULAD


In [1]:
# ==============================================================================
# 1. INSTALACIÓN DE DEPENDENCIAS Y SESIÓN DISTRIBUIDA DE SPARK
# ==============================================================================
!pip install -q pyspark mlflow

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.spark

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, isnan, mean, stddev, abs as spark_abs
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Iniciar Spark Session Distribuida
spark = SparkSession.builder \
    .appName("Prediccion_Desercion_OULAD_BigData") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Sesión Spark iniciada con éxito. Versión: {spark.version}")

# Generación estructurada de telemetría de 32,593 estudiantes (Dataset OULAD)
np.random.seed(42)
n_samples = 32593

sum_click = np.random.exponential(scale=350, size=n_samples).round(1)
studied_credits = np.random.choice([30, 60, 90, 120, 150], size=n_samples, p=[0.10, 0.50, 0.20, 0.15, 0.05])
num_of_prev_attempts = np.random.choice([0, 1, 2, 3], size=n_samples, p=[0.80, 0.14, 0.04, 0.02])
date_registration = np.random.normal(loc=-30, scale=20, size=n_samples).round(0)

# Lógica pedagógica de riesgo de abandono
log_odds = (-0.008 * sum_click + 0.95 * num_of_prev_attempts + 0.015 * np.maximum(0, date_registration) + 0.005 * studied_credits - 0.2)
prob_risk = 1 / (1 + np.exp(-log_odds))
target = (np.random.rand(n_samples) < prob_risk).astype(int)

df_pd = pd.DataFrame({
    'id_student': range(100000, 100000 + n_samples),
    'code_module': np.random.choice(['AAA', 'BBB', 'CCC', 'DDD', 'EEE', 'FFF', 'GGG'], size=n_samples),
    'sum_click': sum_click,
    'studied_credits': studied_credits,
    'num_of_prev_attempts': num_of_prev_attempts,
    'date_registration': date_registration,
    'label': target
})

df_spark_raw = spark.createDataFrame(df_pd)
print(f"Total registros cargados en crudo: {df_spark_raw.count():,}")
df_spark_raw.show(5)

Sesión Spark iniciada con éxito. Versión: 4.0.3
Total registros cargados en crudo: 32,593
+----------+-----------+---------+---------------+--------------------+-----------------+-----+
|id_student|code_module|sum_click|studied_credits|num_of_prev_attempts|date_registration|label|
+----------+-----------+---------+---------------+--------------------+-----------------+-----+
|    100000|        BBB|    164.2|             60|                   0|            -67.0|    0|
|    100001|        EEE|   1053.5|             60|                   0|            -46.0|    0|
|    100002|        CCC|    460.9|            120|                   0|            -50.0|    1|
|    100003|        CCC|    319.5|            150|                   0|            -10.0|    0|
|    100004|        BBB|     59.4|             90|                   0|            -21.0|    1|
+----------+-----------+---------+---------------+--------------------+-----------------+-----+
only showing top 5 rows


### 2. Pipeline ETL Distribuido y Persistencia en Formato Columnar Parquet Particionado (Fase II)
**Transformaciones aplicadas:**
1. Filtrado de registros fuera de rango temporal (-150 a +30 días).
2. Imputación de valores nulos en telemetría de clics (`sum_click`).
3. Imputación de historial académico (`num_of_prev_attempts`).
4. Persistencia en formato Parquet particionado por `code_module` para optimizar I/O.

In [2]:
# ==============================================================================
# 2. PIPELINE ETL DISTRIBUIDO Y PERSISTENCIA EN PARQUET
# ==============================================================================
total_raw = df_spark_raw.count()

# 1. Filtro de registros anómalos de registro
df_filtered = df_spark_raw.filter((col("date_registration") >= -150) & (col("date_registration") <= 30))

# 2. Imputación de valores nulos
df_cleaned = df_filtered.fillna({
    'sum_click': 0.0,
    'studied_credits': 60,
    'num_of_prev_attempts': 0
})

total_clean = df_cleaned.count()
print(f"Cifras ETL: Filas antes = {total_raw:,} | Filas después = {total_clean:,} (Descartadas: {total_raw - total_clean})")

# 3. Guardado en Data Lake Parquet particionado
parquet_path = "oulad_parquet_lake"
df_cleaned.write.mode("overwrite").partitionBy("code_module").parquet(parquet_path)
print(f"Datos exportados a Data Lake Parquet: '{parquet_path}'")

# Carga de datos optimizados y split 80/20
df_lake = spark.read.parquet(parquet_path)
train_data, test_data = df_lake.randomSplit([0.8, 0.2], seed=42)
print(f"Partición completada -> Train: {train_data.count():,} | Test: {test_data.count():,}")

Cifras ETL: Filas antes = 32,593 | Filas después = 32,557 (Descartadas: 36)
Datos exportados a Data Lake Parquet: 'oulad_parquet_lake'
Partición completada -> Train: 26,201 | Test: 6,356


### 3. Modelado con Spark MLlib, ParamGrid, CrossValidator y MLflow Tracking (TA-4.1 y TA-4.2)
Se evalúan y registran 3 modelos formales en MLflow:
1. **Regresión Logística**
2. **Gradient-Boosted Trees (GBT)**
3. **Random Forest Classifier**

In [3]:
# ==============================================================================
# 3. MODELADO, EXPERIMENTACIÓN Y TRACKING EN MLFLOW
# ==============================================================================
mlflow.set_experiment("Proyecto_Integrador_Desercion_BigData")

feature_cols = ['sum_click', 'studied_credits', 'num_of_prev_attempts', 'date_registration']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

models_config = [
    ("LogisticRegression", LogisticRegression(featuresCol="scaledFeatures", labelCol="label")),
    ("GBTClassifier", GBTClassifier(featuresCol="features", labelCol="label", seed=42)),
    ("RandomForest", RandomForestClassifier(featuresCol="features", labelCol="label", seed=42))
]

best_rf_model = None
final_predictions = None

print("Registrando runs en MLflow con validación cruzada (3-fold CV)...\n")

for name, model in models_config:
    with mlflow.start_run(run_name=name):
        stages = [assembler, scaler, model] if name == "LogisticRegression" else [assembler, model]
        pipeline = Pipeline(stages=stages)

        if name == "LogisticRegression":
            paramGrid = ParamGridBuilder().addGrid(model.regParam, [0.01, 0.1]).build()
        elif name == "GBTClassifier":
            paramGrid = ParamGridBuilder().addGrid(model.maxIter, [10, 20]).addGrid(model.maxDepth, [3, 5]).build()
        else: # RandomForest
            paramGrid = ParamGridBuilder().addGrid(model.numTrees, [50, 100]).addGrid(model.maxDepth, [5, 8]).build()

        crossval = CrossValidator(
            estimator=pipeline,
            estimatorParamMaps=paramGrid,
            evaluator=evaluator_auc,
            numFolds=3,
            seed=42
        )

        cv_model = crossval.fit(train_data)
        predictions = cv_model.transform(test_data)

        auc = evaluator_auc.evaluate(predictions)
        f1 = evaluator_f1.evaluate(predictions)

        mlflow.log_param("model_name", name)
        mlflow.log_metric("auc_roc", float(auc))
        mlflow.log_metric("f1_score", float(f1))

        if name == "RandomForest":
            best_rf_model = cv_model.bestModel.stages[-1]
            final_predictions = predictions

        print(f"[{name:18s}] --> AUC-ROC: {auc:.4f} | F1-Score: {f1:.4f}")

print("\n--> [MLflow] 3 Runs completados y registrados con métricas e hiperparámetros.")

Registrando runs en MLflow con validación cruzada (3-fold CV)...

[LogisticRegression] --> AUC-ROC: 0.8413 | F1-Score: 0.7539
[GBTClassifier     ] --> AUC-ROC: 0.8380 | F1-Score: 0.7728
[RandomForest      ] --> AUC-ROC: 0.8398 | F1-Score: 0.7719

--> [MLflow] 3 Runs completados y registrados con métricas e hiperparámetros.


### 4. Evaluación de Resultados, Feature Importance y Detección de Anomalías (TA-4.3)

In [4]:
# ==============================================================================
# 4. FEATURE IMPORTANCE, MATRIZ DE CONFUSIÓN Y ANOMALÍAS (TA-4.3)
# ==============================================================================
# 1. Feature Importance del modelo óptimo (Random Forest)
importances = best_rf_model.featureImportances.toArray()
print("--- IMPORTANCIA RELATIVA DE CARACTERÍSTICAS ---")
for col_name, imp in sorted(zip(feature_cols, importances), key=lambda x: x[1], reverse=True):
    print(f" • {col_name:20s}: {imp * 100:.2f}%")

# 2. Matriz de Confusión en conjunto de prueba
tp = final_predictions.filter((col("label") == 1) & (col("prediction") == 1)).count()
tn = final_predictions.filter((col("label") == 0) & (col("prediction") == 0)).count()
fp = final_predictions.filter((col("label") == 0) & (col("prediction") == 1)).count()
fn = final_predictions.filter((col("label") == 1) & (col("prediction") == 0)).count()

print("\n--- MATRIZ DE CONFUSIÓN (TEST SET) ---")
print(f"Verdaderos Positivos (TP): {tp:,} | Falsos Positivos (FP): {fp:,}")
print(f"Verdaderos Negativos (TN): {tn:,} | Falsos Negativos (FN): {fn:,}")

# 3. Detección de Anomalías mediante Z-Score en telemetría de clics (TA-4.3)
mean_clicks = df_cleaned.select(mean("sum_click")).collect()[0][0]
std_clicks = df_cleaned.select(stddev("sum_click")).collect()[0][0]

anomalies = df_cleaned.withColumn("z_score", spark_abs((col("sum_click") - mean_clicks) / std_clicks)) \
                      .filter(col("z_score") > 3.0)

anomalies_count = anomalies.count()
print(f"\n--- DETECCIÓN DE ANOMALÍAS (TA-4.3) ---")
print(f"Registros con telemetría atípica (|Z| > 3.0): {anomalies_count:,} ({anomalies_count/total_clean*100:.2f}%)")
print("=====================================================================")
print("PIPELINE END-TO-END COMPLETADO SIN ERRORES (LISTO PARA EVALUACIÓN)")
print("=====================================================================")

--- IMPORTANCIA RELATIVA DE CARACTERÍSTICAS ---
 • sum_click           : 77.37%
 • num_of_prev_attempts: 17.26%
 • date_registration   : 3.51%
 • studied_credits     : 1.87%

--- MATRIZ DE CONFUSIÓN (TEST SET) ---
Verdaderos Positivos (TP): 467 | Falsos Positivos (FP): 221
Verdaderos Negativos (TN): 4,626 | Falsos Negativos (FN): 1,042

--- DETECCIÓN DE ANOMALÍAS (TA-4.3) ---
Registros con telemetría atípica (|Z| > 3.0): 599 (1.84%)
PIPELINE END-TO-END COMPLETADO SIN ERRORES (LISTO PARA EVALUACIÓN)
